# Preprocess the CICMalDroid dataset

1. Imports

In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import shuffle

# 1. Load Data
print("--- Loading dataset ---")
DATASET_PATH = '/home/simon/AI_cybersec_lab/Assigment/AI for Cyber/test_submission/cicmaldroid.csv' # set the location for the dataset
df = pd.read_csv(DATASET_PATH)

print(f"Original dataset shape: {df.shape}")
display(df.head())

--- Loading dataset ---
Original dataset shape: (11598, 471)


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,utimes,vfork,vibrate,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev,Class
0,1,0,0,3,0,14,2,0,3,0,...,0,0,0,0,0,0,0,37,10,1
1,3,0,0,6,0,42,91,0,32,0,...,0,0,0,0,0,0,2,2838,46,1
2,2,0,0,4,0,23,3,0,17,2,...,0,0,0,0,0,0,1,111,20,1
3,1,0,0,4,0,27,9,0,36,0,...,0,0,0,0,0,0,7,987,197,1
4,3,0,0,11,0,18,3,0,16,0,...,0,0,0,0,0,0,1,98,25,1


In [37]:
print("--- Initial cleaning, labeling, and downsampling ---")

# Count and drop duplicate rows
num_duplicates = df.duplicated().sum()
print(f"Found and dropped {num_duplicates} duplicated rows.")
df.drop_duplicates(inplace=True)

# Replace infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop zero-variance (constant) features
constant_features = [col for col in df.columns if df[col].nunique() <= 1]
df.drop(columns=constant_features, inplace=True)
print(f"Dropped {len(constant_features)} constant features.")

# Map labels (Required now so we can stratify during the split)
df['Class'] = df['Class'].astype(str).str.strip()
df['Label'] = df['Class'].apply(lambda x: 0 if x == '5' else 1)
df.drop(columns=['Class'], inplace=True)

# Downsample the ENTIRE dataset before splitting
print("\n--- Downsampling entire dataset ---")
df_malware = df[df['Label'] == 1]
df_benign = df[df['Label'] == 0]

n_malware = min(2000, len(df_malware))
n_benign = min(1600, len(df_benign))

df_malware_sampled = df_malware.sample(n=n_malware, random_state=42)
df_benign_sampled = df_benign.sample(n=n_benign, random_state=42)

# Recombine and shuffle
df = pd.concat([df_malware_sampled, df_benign_sampled])
df = shuffle(df, random_state=42).reset_index(drop=True)

print(f"Current dataset shape after downsampling: {df.shape}")
print("Total Label Distribution (1=Malware, 0=Benign):")
print(df['Label'].value_counts())



--- Initial cleaning, labeling, and downsampling ---
Found and dropped 72 duplicated rows.
Dropped 0 constant features.

--- Downsampling entire dataset ---
Current dataset shape after downsampling: (3600, 471)
Total Label Distribution (1=Malware, 0=Benign):
Label
1    2000
0    1600
Name: count, dtype: int64


In [38]:
print("--- Splitting data ---")

X = df.drop(columns=['Label'])
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

--- Splitting data ---
X_train shape: (2520, 470)
y_train shape: (2520,)
X_test shape:  (1080, 470)
y_test shape:  (1080,)


In [39]:
print("--- Handling missing values (fitting on train) ---")

# Identify columns with > 30% missing values in TRAIN
missing_ratios = X_train.isnull().mean()
cols_to_keep = missing_ratios[missing_ratios <= 0.3].index
cols_dropped = len(X_train.columns) - len(cols_to_keep)

# Drop those columns from TRAIN
X_train = X_train[cols_to_keep]

# Fit Imputer: Calculate medians from TRAIN only
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
train_medians = X_train[numeric_cols].median()

# Apply imputation to TRAIN
X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)

print(f"Dropped {cols_dropped} columns with >30% missing values.")
print(f"Missing values remaining in X_train: {X_train.isna().sum().sum()}")
print(f"Current X_train shape: {X_train.shape}")

--- Handling missing values (fitting on train) ---
Dropped 0 columns with >30% missing values.
Missing values remaining in X_train: 0
Current X_train shape: (2520, 470)


In [40]:
print("--- Applying missing value logic to test ---")

# Drop the exact same columns that were dropped in TRAIN
X_test = X_test[cols_to_keep]

# Apply the TRAIN medians to TEST
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)

print(f"Missing values remaining in X_test: {X_test.isna().sum().sum()}")
print(f"Current X_test shape: {X_test.shape}")

--- Applying missing value logic to test ---
Missing values remaining in X_test: 0
Current X_test shape: (1080, 470)


In [41]:
print("--- Scaling features ---")

scaler = MinMaxScaler(feature_range=(0, 1))

# Fit on TRAIN, apply to TRAIN and TEST
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to preserve column names
X_train_final = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_final = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Scaling complete! Transformed Train data preview:")
display(X_train_final.head())

--- Scaling features ---
Scaling complete! Transformed Train data preview:


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,updateServiceLocation,utimes,vfork,vibrate,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev
0,0.000429,0.0,0.0,0.008571,0.007576,0.156342,0.001427,0.004878,0.009189,0.002517,...,0.0,0.0,0.0,0.0,0.0,0.003891,0.0,0.071429,0.010753,0.000634
1,0.000286,0.0,0.0,0.008571,0.007576,0.126844,0.001314,0.004878,0.008473,0.001678,...,0.0,0.0,0.0,0.0,0.0,0.003891,0.0,0.071429,0.014712,0.000423
2,0.000000,0.0,0.0,0.004286,0.000000,0.026549,0.000075,0.000000,0.000358,0.002517,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000320,0.000028
3,0.000429,0.0,0.0,0.025714,0.007576,0.221239,0.001540,0.004878,0.013246,0.006711,...,0.0,0.0,0.0,0.0,0.0,0.003891,0.0,0.107143,0.073593,0.012392
4,0.000429,0.0,0.0,0.028571,0.000000,0.129794,0.014832,0.000000,0.045346,0.063758,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.178571,0.053632,0.001857


In [42]:
print("--- Exporting processed files ---")

# Reattach labels and reset indices to ensure alignment
train_ready = pd.concat([X_train_final, y_train.reset_index(drop=True)], axis=1)
test_ready = pd.concat([X_test_final, y_test.reset_index(drop=True)], axis=1)

# Save to CSV
train_ready.to_csv('cicmaldroid_train_ready.csv', index=False)
test_ready.to_csv('cicmaldroid_test_ready.csv', index=False)

print("Preprocessing complete!")
print(f"Saved 'cicmaldroid_train_ready.csv' with shape {train_ready.shape}")
print(f"Saved 'cicmaldroid_test_ready.csv' with shape {test_ready.shape}")

--- Exporting processed files ---
Preprocessing complete!
Saved 'cicmaldroid_train_ready.csv' with shape (2520, 471)
Saved 'cicmaldroid_test_ready.csv' with shape (1080, 471)


# Build and Evaluate the model architecture

In [43]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU


In [44]:
x = torch.tensor([1.0, 2.0]).cuda()
print(x.device)

cuda:0


In [45]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report
import os
import random

# ==========================================
# REPRODUCIBILITY SEED
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Random seed set to {SEED} for full reproducibility.")
SAVE_PATH = "./dl_amdet_static_weights.pth" # Change this 

Random seed set to 42 for full reproducibility.


In [46]:
# ==========================================
# 1. MODEL DEFINITION
# ==========================================
def build_static_malware_detector(input_features, num_classes=2):
    """
    Constructs the CNN-BiLSTM static analysis model.
    The input_features parameter is determined dynamically from the CSV file.
    """
    # CNN Block
    conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
    pool1 = nn.MaxPool1d(kernel_size=2)
    
    conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
    pool2 = nn.MaxPool1d(kernel_size=2)
    
    conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
    
    # BiLSTM Block
    bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
    
    # Fully Connected Block
    fc1 = nn.Linear(in_features=256, out_features=256)
    dropout = nn.Dropout(p=0.2)
    fc2 = nn.Linear(in_features=256, out_features=num_classes)

    model = nn.ModuleList([conv1, pool1, conv2, pool2, conv3, bilstm, fc1, dropout, fc2])

    def forward(x):
        # Add channel dimension: (batch_size, 1, input_features)
        x = x.unsqueeze(1)
        
        x = pool1(torch.relu(conv1(x)))
        x = pool2(torch.relu(conv2(x)))
        x = torch.relu(conv3(x))
        
        x = x.permute(0, 2, 1) # (batch, channels, seq_len) -> (batch, seq_len, channels)
        
        _, (hidden, _) = bilstm(x)
        
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]
        x = torch.cat((hidden_forward, hidden_backward), dim=1)
        
        x = torch.relu(fc1(x))
        x = dropout(x)
        x = fc2(x)
        
        return x

    model.forward = forward
    return model

In [47]:
# ==========================================
# 2. DATA LOADING PIPELINE
# ==========================================
def load_and_prepare_data(train_path, test_path, batch_size=64):
    """
    Loads CSV files, separates features and labels, and creates DataLoaders.
    Assumes the target label is the LAST column in the CSV.
    """
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Extract features (all columns except the last one) and labels (last column)
    X_train_raw = train_df.iloc[:, :-1].values
    y_train_raw = train_df.iloc[:, -1].values
    
    X_test_raw = test_df.iloc[:, :-1].values
    y_test_raw = test_df.iloc[:, -1].values
    
    # Dynamically determine the number of input features
    num_features = X_train_raw.shape[1]
    print(f"Detected {num_features} input features from the dataset.")
    
    # Convert to PyTorch Tensors
    X_train = torch.tensor(X_train_raw, dtype=torch.float32)
    y_train = torch.tensor(y_train_raw, dtype=torch.long)
    
    X_test = torch.tensor(X_test_raw, dtype=torch.float32)
    y_test = torch.tensor(y_test_raw, dtype=torch.long)
    
    # Create DataLoaders
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    
    # Use a seeded generator so shuffle order is identical across runs
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, num_features

In [48]:
# ==========================================
# 3. TRAINING & EVALUATION LOOPS
# ==========================================
def train_model(model, train_loader, epochs=50, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    print(f"--- Starting Training for {epochs} Epochs ---")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        print(f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | Train Acc: {(correct/total)*100:.2f}%")
        
    return model

In [49]:
def evaluate_model(model, test_loader):
    model.eval() # Disable dropout for evaluation
    criterion = nn.CrossEntropyLoss()
    
    test_loss = 0.0
    correct = 0
    total = 0
    
    print("--- Evaluating on Test Set ---")
    with torch.no_grad(): # Disable gradient tracking to save memory/compute
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = (correct / total) * 100
    avg_loss = test_loss / len(test_loader)
    
    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Test Accuracy: {accuracy:.2f}%")

In [50]:
# ==========================================
# 4. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    # Define file paths
    TRAIN_CSV = "cicmaldroid_train_ready.csv"
    TEST_CSV = "cicmaldroid_test_ready.csv"
    
    # Hyperparameters from the paper
    BATCH_SIZE = 64
    EPOCHS = 50
    LEARNING_RATE = 0.001
    
    try:
        # 1. Load data
        train_loader, test_loader, num_features = load_and_prepare_data(
            train_path=TRAIN_CSV, 
            test_path=TEST_CSV, 
            batch_size=BATCH_SIZE
        )
        
        # 2. Build model
        print("Building CNN-BiLSTM model...")
        model = build_static_malware_detector(input_features=num_features)
        
        # 3. Train the model
        trained_model = train_model(
            model=model, 
            train_loader=train_loader, 
            epochs=EPOCHS, 
            lr=LEARNING_RATE
        )
        
        # 4. Evaluate the model
        evaluate_model(trained_model, test_loader)
        
        # ---------------------------------------------------------
        # 5. SAVE THE MODEL PARAMETERS
        # ---------------------------------------------------------

        
        # Save only the state dictionary (weights and biases)
        torch.save(trained_model.state_dict(), SAVE_PATH)
        print(f"\nModel parameters successfully saved to: {os.path.abspath(SAVE_PATH)}")
        
    except FileNotFoundError:
        print(f"Error: Could not find '{TRAIN_CSV}' or '{TEST_CSV}'.")

Loading datasets...
Detected 470 input features from the dataset.
Building CNN-BiLSTM model...
--- Starting Training for 50 Epochs ---
Epoch [01/50] | Loss: 0.6874 | Train Acc: 55.32%
Epoch [02/50] | Loss: 0.6706 | Train Acc: 58.97%
Epoch [03/50] | Loss: 0.6057 | Train Acc: 68.41%
Epoch [04/50] | Loss: 0.5849 | Train Acc: 70.40%
Epoch [05/50] | Loss: 0.5600 | Train Acc: 70.99%
Epoch [06/50] | Loss: 0.5279 | Train Acc: 73.57%
Epoch [07/50] | Loss: 0.5062 | Train Acc: 75.40%
Epoch [08/50] | Loss: 0.4780 | Train Acc: 77.90%
Epoch [09/50] | Loss: 0.4764 | Train Acc: 77.98%
Epoch [10/50] | Loss: 0.4484 | Train Acc: 79.64%
Epoch [11/50] | Loss: 0.4381 | Train Acc: 80.28%
Epoch [12/50] | Loss: 0.4187 | Train Acc: 81.11%
Epoch [13/50] | Loss: 0.4047 | Train Acc: 80.83%
Epoch [14/50] | Loss: 0.3993 | Train Acc: 82.58%
Epoch [15/50] | Loss: 0.3946 | Train Acc: 82.10%
Epoch [16/50] | Loss: 0.3917 | Train Acc: 82.14%
Epoch [17/50] | Loss: 0.3644 | Train Acc: 83.53%
Epoch [18/50] | Loss: 0.3822 | T

In [51]:
# ==========================================
# 2. DATA LOADING (Test Set Only)
# ==========================================
def load_test_data(test_path, batch_size=64):
    print(f"Loading test data from {test_path}...")
    test_df = pd.read_csv(test_path)
    
    # Extract features (all but last col) and labels (last col)
    X_test_raw = test_df.iloc[:, :-1].values
    y_test_raw = test_df.iloc[:, -1].values
    num_features = X_test_raw.shape[1]
    
    # Convert to Tensors
    X_test = torch.tensor(X_test_raw, dtype=torch.float32)
    y_test = torch.tensor(y_test_raw, dtype=torch.long)
    
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return test_loader, num_features

# ==========================================
# 3. EVALUATION FUNCTION
# ==========================================
def evaluate_and_print_metrics(model, test_loader):
    # 1. Set model to evaluation mode (turns off dropout!)
    model.eval()
    
    all_predictions = []
    all_true_labels = []
    
    print("Running predictions on the test set...")
    
    # 2. Disable gradient calculations for faster, memory-efficient inference
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            
            # The model outputs raw logits for 2 classes. 
            # We take the index of the highest logit as the predicted class.
            _, predicted = torch.max(outputs.data, 1)
            
            # Move data off GPU (if using one) and convert to standard python lists
            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    # 3. Calculate Metrics using Scikit-Learn
    # average='binary' assumes your positive class (malware) is labeled as 1
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true_labels, 
        all_predictions, 
        average='binary'
    )
    
    conf_matrix = confusion_matrix(all_true_labels, all_predictions)
    
    # 4. Print the results
    print("\n" + "="*40)
    print("FINAL EVALUATION METRICS")
    print("="*40)
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall:    {recall * 100:.2f}%")
    print(f"F1-Score:  {f1 * 100:.2f}%")
    
    print("\nConfusion Matrix:")
    print("                Predicted Benign (0) | Predicted Malware (1)")
    print(f"True Benign (0):       {conf_matrix[0][0]:<13} | {conf_matrix[0][1]}")
    print(f"True Malware (1):      {conf_matrix[1][0]:<13} | {conf_matrix[1][1]}")
    print("="*40)
    
    # Optional: Print a highly detailed report for both classes
    # print("\nDetailed Classification Report:")
    # print(classification_report(all_true_labels, all_predictions, target_names=["Benign", "Malware"]))

# ==========================================
# 4. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    TEST_CSV = "cicmaldroid_test_ready.csv"
    BATCH_SIZE = 64
    
    try:
        # 1. Load the test dataset and get the feature count
        test_loader, num_features = load_test_data(TEST_CSV, batch_size=BATCH_SIZE)
        
        # 2. Initialize the blank architecture
        model = build_static_malware_detector(input_features=num_features)
        
        # 3. Load the saved weights into the model
        # map_location='cpu' ensures it loads safely even if trained on a GPU and tested on CPU
        model.load_state_dict(torch.load(SAVE_PATH, map_location=torch.device('cpu'), weights_only=True))
        print("Model parameters loaded successfully.")
        
        # 4. Run the evaluation
        evaluate_and_print_metrics(model, test_loader)
        
    except FileNotFoundError as e:
        print(f"Error: Could not find a required file. {e}")

Loading test data from cicmaldroid_test_ready.csv...
Model parameters loaded successfully.
Running predictions on the test set...

FINAL EVALUATION METRICS
Precision: 90.05%
Recall:    84.50%
F1-Score:  87.19%

Confusion Matrix:
                Predicted Benign (0) | Predicted Malware (1)
True Benign (0):       424           | 56
True Malware (1):      93            | 507


# Extended Experiement

## Preprocess the CLaMP dataset

In [52]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer

# Load the data
DATASET_PATH = "./ClaMP_Raw-5184.csv"
df = pd.read_csv(DATASET_PATH)

print("--- Data Loading ---")
print(f"Original dataset shape: {df.shape}")
display(df.head())

--- Data Loading ---
Original dataset shape: (5184, 56)


,e_magic,e_cblp,e_cp,e_crlc,e_cparhdr,e_minalloc,e_maxalloc,e_ss,e_sp,e_csum,...,CheckSum,Subsystem,DllCharacteristics,SizeOfStackReserve,SizeOfStackCommit,SizeOfHeapReserve,SizeOfHeapCommit,LoaderFlags,NumberOfRvaAndSizes,class
0,23117,144,3,0,4,0,65535,0,184,0,...,1194954,3,64,1048576,4096,1048576,4096,0,16,0
1,23117,144,3,0,4,0,65535,0,184,0,...,0,2,0,1048576,4096,1048576,4096,0,16,0
2,23117,144,3,0,4,0,65535,0,184,0,...,67688,2,320,1048576,4096,1048576,4096,0,16,0
3,23117,144,3,0,4,0,65535,0,184,0,...,113668,2,1344,1048576,4096,1048576,4096,0,16,0
4,23117,144,3,0,4,0,65535,0,184,0,...,69089,2,33088,262144,8192,1048576,4096,0,16,0


In [53]:
print("--- 1. Splitting data ---")

# Split chronologically based on CreationYear
train_data = df[df['CreationYear'] < 2010].copy()
test_data = df[df['CreationYear'] >= 2010].copy()

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape:  {test_data.shape}")

--- 1. Splitting data ---
Train data shape: (2035, 56)
Test data shape:  (3149, 56)


In [54]:
print("--- 2. Handling Inf and Null values ---")

# Convert Inf to NaN so they can be handled together
train_data.replace([np.inf, -np.inf], np.nan, inplace=True)
test_data.replace([np.inf, -np.inf], np.nan, inplace=True)

print(f"Total missing values before handling in Train data: {train_data.isna().sum().sum()}")
print(f"Total missing values before handling in Test data:  {test_data.isna().sum().sum()}")

# Calculate median strictly from the TRAIN set to prevent data leakage
fill_values = train_data.median(numeric_only=True)

train_data.fillna(fill_values, inplace=True)
test_data.fillna(fill_values, inplace=True)

print(f"Total missing values after handling in Train data: {train_data.isna().sum().sum()}")
print(f"Total missing values after handling in Test data:  {test_data.isna().sum().sum()}")

--- 2. Handling Inf and Null values ---
Total missing values before handling in Train data: 4070
Total missing values before handling in Test data:  6298
Total missing values after handling in Train data: 4070
Total missing values after handling in Test data:  6298


In [55]:
print("--- 3. Dropping constant columns ---")

# Find columns in the train set that have only 1 unique value
constant_cols = [col for col in train_data.columns if train_data[col].nunique() <= 1]

train_data.drop(columns=constant_cols, inplace=True)
test_data.drop(columns=constant_cols, inplace=True)

print(f"Dropped {len(constant_cols)} constant columns.")
if len(constant_cols) > 0:
    print(f"Example columns dropped: {constant_cols[:5]}")
print(f"Current Train data shape: {train_data.shape}")

--- 3. Dropping constant columns ---
Dropped 8 constant columns.
Example columns dropped: ['e_magic', 'e_crlc', 'e_ss', 'e_res', 'e_res2']
Current Train data shape: (2035, 48)


In [56]:
print("--- 4. Dropping duplicated columns ---")

# Find identical columns based on the Train set
duplicated_cols = set()
for i in range(train_data.shape[1]):
    col1 = train_data.iloc[:, i]
    for j in range(i + 1, train_data.shape[1]):
        col2 = train_data.iloc[:, j]
        if col1.equals(col2):
            duplicated_cols.add(train_data.columns[j])

duplicated_list = list(duplicated_cols)
train_data.drop(columns=duplicated_list, inplace=True)
test_data.drop(columns=duplicated_list, inplace=True)

print(f"Dropped {len(duplicated_list)} duplicated columns.")
if len(duplicated_list) > 0:
    print(f"Example columns dropped: {duplicated_list[:5]}")
print(f"Current Train data shape: {train_data.shape}")

--- 4. Dropping duplicated columns ---
Dropped 0 duplicated columns.
Current Train data shape: (2035, 48)


In [57]:
print("--- 5. Dropping correlated columns (threshold > 0.9) ---")

# Calculate correlation matrix only on the TRAIN set
corr_matrix = train_data.corr(numeric_only=True).abs()

# Select the upper triangle of the correlation matrix
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.9
to_drop_corr = [column for column in upper_tri.columns if any(upper_tri[column] > 0.9)]

train_data.drop(columns=to_drop_corr, inplace=True)
test_data.drop(columns=to_drop_corr, inplace=True)

print(f"Dropped {len(to_drop_corr)} highly correlated columns.")
print(f"Current Train data shape: {train_data.shape}")
print(f"Current Test data shape:  {test_data.shape}")

--- 5. Dropping correlated columns (threshold > 0.9) ---
Dropped 8 highly correlated columns.
Current Train data shape: (2035, 40)
Current Test data shape:  (3149, 40)


In [58]:
print("--- 6. Scaling features ---")

scaler = QuantileTransformer()

# Fit strictly on train data, transform both
train_scaled_array = scaler.fit_transform(train_data)
test_scaled_array = scaler.transform(test_data)

# Convert back to Pandas DataFrames to maintain index and column names
train_final = pd.DataFrame(train_scaled_array, columns=train_data.columns, index=train_data.index)
test_final = pd.DataFrame(test_scaled_array, columns=test_data.columns, index=test_data.index)

print("Pipeline complete! Transformed Train data preview:")
display(train_final.head())

--- 6. Scaling features ---
Pipeline complete! Transformed Train data preview:


,e_cblp,e_cparhdr,e_minalloc,e_maxalloc,e_sp,e_csum,e_ip,e_lfanew,Machine,NumberOfSections,...,SizeOfHeaders,CheckSum,Subsystem,DllCharacteristics,SizeOfStackReserve,SizeOfStackCommit,SizeOfHeapReserve,SizeOfHeapCommit,LoaderFlags,class
0,0.560561,1.0,0.000000,1.0,0.498999,0.0,0.0,0.821822,0.0,0.437437,...,0.327327,0.932469,0.943944,0.748248,0.451952,0.336336,0.431431,0.419419,0.0,0.0
1,0.560561,1.0,0.000000,1.0,0.498999,0.0,0.0,0.168669,0.0,0.437437,...,0.327327,0.000000,0.444444,0.000000,0.451952,0.336336,0.431431,0.419419,0.0,0.0
5,0.059059,1.0,0.936436,1.0,0.498999,0.0,0.0,0.821822,0.0,0.959960,...,0.327327,0.987433,0.444444,0.000000,0.451952,0.821822,0.431431,0.419419,0.0,0.0
10,0.560561,1.0,0.000000,1.0,0.498999,0.0,0.0,0.663664,0.0,0.437437,...,0.864364,0.000000,0.444444,0.000000,0.451952,0.336336,0.431431,0.419419,0.0,0.0
12,0.560561,1.0,0.000000,1.0,0.498999,0.0,0.0,0.230230,0.0,0.165165,...,0.864364,0.692738,0.444444,0.000000,0.451952,0.336336,0.431431,0.419419,0.0,0.0


In [59]:
print("--- 7. Exporting datasets to CSV ---")

# Save the training set
train_final.to_csv('CLaMP_train_data.csv', index=False)

# Save the testing set
test_final.to_csv('CLaMP_test_data.csv', index=False)

print("Export complete!")
print("Files successfully saved as 'CLaMP_train_data.csv' and 'CLaMP_test_data.csv'.")

--- 7. Exporting datasets to CSV ---
Export complete!
Files successfully saved as 'CLaMP_train_data.csv' and 'CLaMP_test_data.csv'.


## Build and evaluate the model for the extended experiment

In [60]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU


In [61]:
x = torch.tensor([1.0, 2.0]).cuda()
print(x.device)

cuda:0


In [62]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report
import os
import random

# ==========================================
# REPRODUCIBILITY SEED
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Random seed set to {SEED} for full reproducibility.")

Random seed set to 42 for full reproducibility.


In [63]:
# ==========================================
# 1. MODEL DEFINITION
# ==========================================
def build_static_malware_detector(input_features, num_classes):
    """
    Constructs the CNN-BiLSTM static analysis model.
    The input_features parameter is determined dynamically from the CSV file.
    """
    # CNN Block
    conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
    pool1 = nn.MaxPool1d(kernel_size=2)
    
    conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
    pool2 = nn.MaxPool1d(kernel_size=2)
    
    conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
    
    # BiLSTM Block
    bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
    
    # Fully Connected Block
    fc1 = nn.Linear(in_features=256, out_features=256)
    dropout = nn.Dropout(p=0.2)
    fc2 = nn.Linear(in_features=256, out_features=num_classes)

    model = nn.ModuleList([conv1, pool1, conv2, pool2, conv3, bilstm, fc1, dropout, fc2])

    def forward(x):
        # Add channel dimension: (batch_size, 1, input_features)
        x = x.unsqueeze(1)
        
        x = pool1(torch.relu(conv1(x)))
        x = pool2(torch.relu(conv2(x)))
        x = torch.relu(conv3(x))
        
        x = x.permute(0, 2, 1) # (batch, channels, seq_len) -> (batch, seq_len, channels)
        
        _, (hidden, _) = bilstm(x)
        
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]
        x = torch.cat((hidden_forward, hidden_backward), dim=1)
        
        x = torch.relu(fc1(x))
        x = dropout(x)
        x = fc2(x)
        
        return x

    model.forward = forward
    return model

In [64]:
# ==========================================
# 2. DATA LOADING PIPELINE
# ==========================================
def load_and_prepare_data(train_path, test_path, batch_size=64):
    """
    Loads CSV files, separates features and labels, and creates DataLoaders.
    Assumes the target label is the LAST column in the CSV.
    """
    print("Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Extract features (all columns except the last one) and labels (last column)
    X_train_raw = train_df.iloc[:, :-1].values
    y_train_raw = train_df.iloc[:, -1].values
    
    X_test_raw = test_df.iloc[:, :-1].values
    y_test_raw = test_df.iloc[:, -1].values
    
    # Dynamically determine the number of input features
    num_features = X_train_raw.shape[1]
    
    # Dynamically count how many unique classes exist in the training labels
    num_classes = len(np.unique(y_train_raw))
    
    print(f"Detected {num_features} input features and {num_classes} classes.")
    
    # Convert to PyTorch Tensors
    X_train = torch.tensor(X_train_raw, dtype=torch.float32)
    y_train = torch.tensor(y_train_raw, dtype=torch.long)
    
    X_test = torch.tensor(X_test_raw, dtype=torch.float32)
    y_test = torch.tensor(y_test_raw, dtype=torch.long)
    
    # Create DataLoaders
    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)
    
    # Use a seeded generator so shuffle order is identical across runs
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, num_features, num_classes

In [65]:
# ==========================================
# 3. TRAINING & EVALUATION LOOPS
# ==========================================
def train_model(model, train_loader, epochs=50, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    print(f"--- Starting Training for {epochs} Epochs ---")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        print(f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | Train Acc: {(correct/total)*100:.2f}%")
        
    return model

In [66]:
def evaluate_model(model, test_loader):
    model.eval() # Disable dropout for evaluation
    criterion = nn.CrossEntropyLoss()
    
    test_loss = 0.0
    correct = 0
    total = 0
    
    print("--- Evaluating on Test Set ---")
    with torch.no_grad(): # Disable gradient tracking to save memory/compute
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = (correct / total) * 100
    avg_loss = test_loss / len(test_loader)
    
    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Test Accuracy: {accuracy:.2f}%")

In [67]:
# ==========================================
# 4. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    # Define file paths
    TRAIN_CSV = "CLaMP_train_data.csv"
    TEST_CSV = "CLaMP_test_data.csv"
    
    # Hyperparameters from the paper
    BATCH_SIZE = 64
    EPOCHS = 50
    LEARNING_RATE = 0.001
    
    try:
        # 1. Load data
# 1. Load data (NEW: unpack num_classes)
        train_loader, test_loader, num_features, num_classes = load_and_prepare_data(
            train_path=TRAIN_CSV, 
            test_path=TEST_CSV, 
            batch_size=BATCH_SIZE
        )
        
        # 2. Build model (NEW: pass num_classes into the model builder)
        print("Building CNN-BiLSTM model...")
        model = build_static_malware_detector(input_features=num_features, num_classes=num_classes)
        
        
        # 3. Train the model
        trained_model = train_model(
            model=model, 
            train_loader=train_loader, 
            epochs=EPOCHS, 
            lr=LEARNING_RATE
        )
        
        # 4. Evaluate the model
        evaluate_model(trained_model, test_loader)
        
        # ---------------------------------------------------------
        # 5. SAVE THE MODEL PARAMETERS
        # ---------------------------------------------------------
        SAVE_PATH = "./dl_amdet_static_weights_CLaMP.pth"
        
        # Save only the state dictionary (weights and biases)
        torch.save(trained_model.state_dict(), SAVE_PATH)
        print(f"\nModel parameters successfully saved to: {os.path.abspath(SAVE_PATH)}")
        
    except FileNotFoundError:
        print(f"Error: Could not find '{TRAIN_CSV}' or '{TEST_CSV}'.")

Loading datasets...
Detected 39 input features and 2 classes.
Building CNN-BiLSTM model...
--- Starting Training for 50 Epochs ---
Epoch [01/50] | Loss: 0.6412 | Train Acc: 64.57%
Epoch [02/50] | Loss: 0.4208 | Train Acc: 80.49%
Epoch [03/50] | Loss: 0.3144 | Train Acc: 86.00%
Epoch [04/50] | Loss: 0.2675 | Train Acc: 88.99%
Epoch [05/50] | Loss: 0.2509 | Train Acc: 90.27%
Epoch [06/50] | Loss: 0.1927 | Train Acc: 92.33%
Epoch [07/50] | Loss: 0.1937 | Train Acc: 92.68%
Epoch [08/50] | Loss: 0.1446 | Train Acc: 94.79%
Epoch [09/50] | Loss: 0.1391 | Train Acc: 93.96%
Epoch [10/50] | Loss: 0.1204 | Train Acc: 95.68%
Epoch [11/50] | Loss: 0.0968 | Train Acc: 96.17%
Epoch [12/50] | Loss: 0.0893 | Train Acc: 96.61%
Epoch [13/50] | Loss: 0.0796 | Train Acc: 96.66%
Epoch [14/50] | Loss: 0.0901 | Train Acc: 96.46%
Epoch [15/50] | Loss: 0.0828 | Train Acc: 96.61%
Epoch [16/50] | Loss: 0.0724 | Train Acc: 97.20%
Epoch [17/50] | Loss: 0.0653 | Train Acc: 97.30%
Epoch [18/50] | Loss: 0.0592 | Train

In [68]:
# ==========================================
# 2. DATA LOADING (Test Set Only)
# ==========================================
def load_test_data(test_path, batch_size=64):
    print(f"Loading test data from {test_path}...")
    test_df = pd.read_csv(test_path)
    
    # Extract features (all but last col) and labels (last col)
    X_test_raw = test_df.iloc[:, :-1].values
    y_test_raw = test_df.iloc[:, -1].values
    num_features = X_test_raw.shape[1]
    
    # Convert to Tensors
    X_test = torch.tensor(X_test_raw, dtype=torch.float32)
    y_test = torch.tensor(y_test_raw, dtype=torch.long)
    
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return test_loader, num_features

# ==========================================
# 3. EVALUATION FUNCTION
# ==========================================
def evaluate_and_print_metrics(model, test_loader):
    # 1. Set model to evaluation mode (turns off dropout!)
    model.eval()
    
    all_predictions = []
    all_true_labels = []
    
    print("Running predictions on the test set...")
    
    # 2. Disable gradient calculations for faster, memory-efficient inference
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            
            # The model outputs raw logits for 2 classes. 
            # We take the index of the highest logit as the predicted class.
            _, predicted = torch.max(outputs.data, 1)
            
            # Move data off GPU (if using one) and convert to standard python lists
            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    # 3. Calculate Metrics using Scikit-Learn
    # average='binary' assumes your positive class (malware) is labeled as 1
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true_labels, 
        all_predictions, 
        average='binary'
    )
    
    conf_matrix = confusion_matrix(all_true_labels, all_predictions)
    
    # 4. Print the results
    print("\n" + "="*40)
    print("FINAL EVALUATION METRICS")
    print("="*40)
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall:    {recall * 100:.2f}%")
    print(f"F1-Score:  {f1 * 100:.2f}%")
    
    print("\nConfusion Matrix:")
    print("                Predicted Benign (0) | Predicted Malware (1)")
    print(f"True Benign (0):       {conf_matrix[0][0]:<13} | {conf_matrix[0][1]}")
    print(f"True Malware (1):      {conf_matrix[1][0]:<13} | {conf_matrix[1][1]}")
    print("="*40)
    
    # Optional: Print a highly detailed report for both classes
    # print("\nDetailed Classification Report:")
    # print(classification_report(all_true_labels, all_predictions, target_names=["Benign", "Malware"]))

# ==========================================
# 4. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    TEST_CSV = "CLaMP_test_data.csv"
    WEIGHTS_PATH = "./dl_amdet_static_weights_CLaMP.pth"
    BATCH_SIZE = 64
    
    try:
        # 1. Load the test dataset and get the feature count
        test_loader, num_features = load_test_data(TEST_CSV, batch_size=BATCH_SIZE)
        
        # 2. Initialize the blank architecture
        model = build_static_malware_detector(input_features=num_features, num_classes=num_classes)
        
        # 3. Load the saved weights into the model
        # map_location='cpu' ensures it loads safely even if trained on a GPU and tested on CPU
        model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=torch.device('cpu'), weights_only=True))
        print("Model parameters loaded successfully.")
        
        # 4. Run the evaluation
        evaluate_and_print_metrics(model, test_loader)
        
    except FileNotFoundError as e:
        print(f"Error: Could not find a required file. {e}")

Loading test data from CLaMP_test_data.csv...
Model parameters loaded successfully.
Running predictions on the test set...

FINAL EVALUATION METRICS
Precision: 97.29%
Recall:    66.92%
F1-Score:  79.30%

Confusion Matrix:
                Predicted Benign (0) | Predicted Malware (1)
True Benign (0):       1566          | 29
True Malware (1):      514           | 1040
